# Weinberg Data 
(1/2 peak potentials as oxidation potentials, needs to be cleansed and ref corrected, compound names but no smiles strings)

In [ ]:
import pandas as pd
Weinberg_Data_Raw = pd.read_excel('../Datasets/Raw Dataset/Weinberg_Data_Raw.xlsx')

In [ ]:
print(Weinberg_Data_Raw)

In [ ]:
import pandas as pd

# Drop rows where 'E1/2 V' contains 'Ep/2, Ep, Ed'
# Drop 'near/Near' from string - just take the value
Weinberg_Data_Filtered = Weinberg_Data_Raw[
    ~Weinberg_Data_Raw['E1/2 (V)'].astype(str).str.contains('Ep/2|Ep|Ed', na=False)
].copy()

Weinberg_Data_Filtered['E1/2 (V)'] = (
    Weinberg_Data_Filtered['E1/2 (V)']
    .astype(str)
    .str.replace(r'(?i)near', '', regex=True) 
    .str.strip()
)

print(f"Original rows: {len(Weinberg_Data_Raw)}")
print(f"Filtered rows: {len(Weinberg_Data_Filtered)}")

In [ ]:
# Remove additional characters from oxidation potential column
df = Weinberg_Data_Filtered.copy()

df['E1/2 (V)'] = (
    df['E1/2 (V)']
    .astype(str)  # make sure it's string
    .str.replace(r"[^0-9\.\,\-\+]", "", regex=True)  # drop unwanted chars
    .str.strip()  # remove leading/trailing spaces
)

In [ ]:
# When there are many values reported, take the first value (first oxidation wave potential)
df['E1/2 (V)'] = df['E1/2 (V)'].astype(str)

# Function: return first numeric value if available
def get_first_value(val):
    try:
        nums = [float(x) for x in val.split(',') if x.strip() != ""]
        return nums[0] if len(nums) > 0 else np.nan
    except:
        return np.nan

# Replace column with first value
df['E1/2 (V)'] = df['E1/2 (V)'].apply(get_first_value)

In [ ]:
# Only keep common reference electrodes
allowed_refs = ['x', 'y', 'z' ,'n', 's', 'f']  # lowercase for comparison

df = df[df['Ref Electrode'].str.lower().isin(allowed_refs)]

print(df['Ref Electrode'].unique())
print(len(df))

In [ ]:
import pandas as pd

# Convert measured potentials to potential vs SHE
ref_to_she = {
    'x': 0.241,
    'y': 0.8,
    'z': 0.8,
    'n': -0.006,
    's': 0.280,
    'f': 0.197
}

df['Ref Electrode'] = df['Ref Electrode'].astype(str)

df['offset'] = df['Ref Electrode'].str.lower().map(ref_to_she)

df['E1/2 (SHE) (V)'] = df['E1/2 (V)'].astype(float) + df['offset']

df = df.drop(columns=['offset'])

print(df[['E1/2 (V)', 'Ref Electrode', 'E1/2 (SHE) (V)']].head())

In [ ]:
import pubchempy as pcp
from tqdm import tqdm
import requests

# Convert IUPAC names to SMILES strings using Pubchem and OPSIN
def name_to_smiles(name):
    """Try to convert a compound name to SMILES using PubChem then OPSIN."""
    name = str(name).strip()
    if not name or name.lower() in ["nan", "none"]:
        return None
    
    try:
        compounds = pcp.get_compounds(name, 'iupac_name')
        if compounds:
            return compounds[0].connectivity_smiles
    except Exception:
        pass
    
    try:
        url = f"https://opsin.ch.cam.ac.uk/opsin/{name}.json"
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            data = r.json()
            if "smiles" in data:
                return data["smiles"]
    except Exception:
        pass
    
    return None


def add_smiles_column(df, name_col="Compound"):
    """Append SMILES column to dataframe with tqdm progress and failure count."""
    failed = 0
    smiles_list = []
    
    for name in tqdm(df[name_col], desc="Converting names to SMILES"):
        smi = name_to_smiles(name)
        if smi is None:
            failed += 1
        smiles_list.append(smi)
    
    df["SMILES"] = smiles_list
    print(f"❌ Failed conversions: {failed}/{len(df)}")
    print(f"✅ Successful conversions: {len(df) - failed}/{len(df)}")
    return df


df_with_smiles = add_smiles_column(df, name_col="Compound")

In [ ]:
print(df_with_smiles[df_with_smiles['SMILES'].isna()])

In [ ]:
import pandas as pd
from rdkit import Chem
from tqdm import tqdm
# Canonicalize those smiles strings
def add_canonical_smiles(df, smiles_col="SMILES", new_col="canonical_SMILES"):
    canonicalized = []

    for smi in tqdm(df[smiles_col], desc="Canonicalizing SMILES"):
        if pd.isna(smi) or smi == "None":
            canonicalized.append(None)
            continue
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                canonicalized.append(None)
            else:
                can_smi = Chem.MolToSmiles(mol, canonical=True)
                canonicalized.append(can_smi)
        except Exception:
            canonicalized.append(None)

    df[new_col] = canonicalized
    return df

df_with_smiles = df_with_smiles.dropna(subset=['SMILES'])
df_with_smiles = add_canonical_smiles(df_with_smiles, "SMILES", "canonical_SMILES")


In [ ]:
df_with_smiles.to_csv('../Datasets/Curated Dataset(s)/Weinberg_w_SMILES.csv')

# Gao Data
Some compounds are being oxidized (quinones), others reduced (phenazines). We will keep only the quinones

In [ ]:
Gao_Data_Raw = pd.read_excel('../Datasets/Raw Dataset/Gao_Data_Raw.xlsx')

In [ ]:
# SMILES strings already exist - canonicalize them
Gao_Data_Raw = add_canonical_smiles(Gao_Data_Raw, "SMILES", "canonical_SMILES")

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
# Group molecules by their type - carbonyl in ring (likely quinones), N in ring (likely phenazines)
# SMARTS patterns
carbonyl_in_ring = Chem.MolFromSmarts("[#6R](=O)")  # ring C=O-O
nitrogen_in_ring = Chem.MolFromSmarts("[#7R]")  # ring N

def classify_smiles(smiles):
    """Classify molecule into one of four groups."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Invalid"

    has_carbonyl = mol.HasSubstructMatch(carbonyl_in_ring)
    has_ringN = mol.HasSubstructMatch(nitrogen_in_ring)

    if has_carbonyl and has_ringN:
        return "Both"
    elif has_carbonyl:
        return "Carbonyl_in_ring"
    elif has_ringN:
        return "Nitrogen_in_ring"
    else:
        return "Neither"

# Load dataset
Gao_Data_Raw["Group"] = Gao_Data_Raw["canonical_SMILES"].apply(classify_smiles)

# Show group counts
print(Gao_Data_Raw["Group"].value_counts())

# Draw 5 samples from each group
sample_mols = {}
for group in Gao_Data_Raw["Group"].unique():
    subset = Gao_Data_Raw[Gao_Data_Raw["Group"] == group].sample(n=min(3, len(Gao_Data_Raw[Gao_Data_Raw["Group"] == group])))
    mols = [Chem.MolFromSmiles(smi) for smi in subset["canonical_SMILES"] if Chem.MolFromSmiles(smi)]
    sample_mols[group] = mols

for group, mols in sample_mols.items():
    if mols:
        img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(200,200),
                                   legends=[group]*len(mols))
        display(img)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Visualize Oxidation Potential by Group
plt.figure(figsize=(8,6))
sns.kdeplot(data=Gao_Data_Raw, x="Redox Potential(V)", hue="Group", fill=True, common_norm=False, alpha=0.4)
plt.title("Oxidation Potential Density by Group")
plt.xlabel("Redox Potential(V)")
plt.ylabel("Density")
plt.tight_layout()
plt.show()


In [ ]:
def get_valence_violations(mol):
    """
    Return a list of atom indices that violate explicit valence rules
    for C, O, H, N, S.
    """
    violations = []
    mol.UpdatePropertyCache(strict=False)
    for atom in mol.GetAtoms():
        idx = atom.GetIdx()
        atomic_num = atom.GetAtomicNum()
        valence = atom.GetTotalValence()
        n_rad = atom.GetNumRadicalElectrons()
        total = valence + n_rad

        if atomic_num == 1:  # H
            if total != 1:
                violations.append(idx)
        elif atomic_num == 6:  # C
            if total > 4:
                violations.append(idx)
        elif atomic_num == 7:  # N
            if total > 4:
                violations.append(idx)
        elif atomic_num == 8:  # O
            if total > 2:
                violations.append(idx)
        elif atomic_num == 16:  # S
            if total not in [2, 4, 6]:
                violations.append(idx)
    return violations
    
from collections import deque
from rdkit import Chem
def reduce_quinone_stepwise_with_radicals(smiles):
    # -------------------
    # Build original molecule
    # -------------------
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    before = Chem.Mol(mol)
    emol = Chem.RWMol(mol)

    # -------------------
    # Step 1: find aromatic carbonyls
    # -------------------
    carbonyls = []
    for o in emol.GetAtoms():
        if o.GetAtomicNum() != 8:
            continue
    
        for c in o.GetNeighbors():
            if c.GetAtomicNum() != 6:
                continue
                
            if c.IsInRing() != True:
                continue
    
            bond = emol.GetBondBetweenAtoms(o.GetIdx(), c.GetIdx())
            if not bond or bond.GetBondType() != Chem.BondType.DOUBLE:
                continue
    
            # Check if carbonyl carbon has at least one conjugated bond
            carbon_conjugated = any(b.GetIsConjugated() for b in c.GetBonds())
            if not carbon_conjugated:
                continue
    
            # Check if carbonyl carbon has a conjugated neighbor (other than O)
            conjugated_neighbor = False
            for nbr in c.GetNeighbors():
                if nbr.GetIdx() == o.GetIdx():
                    continue
                if any(b.GetIsConjugated() for b in nbr.GetBonds()) and nbr.GetAtomicNum() in [6, 7]:
                    conjugated_neighbor = True
                    break
    
            if conjugated_neighbor:
                carbonyls.append((c.GetIdx(), o.GetIdx()))
                    
    if not carbonyls:
        raise ValueError("No aromatic carbonyls found")

    # -------------------
    # Step 2–3: reduce each carbonyl to OH + radical C
    # -------------------
    for c_idx, o_idx in carbonyls:
        # Add H to oxygen
        h_idx = emol.AddAtom(Chem.Atom("H"))
        emol.AddBond(o_idx, h_idx, Chem.BondType.SINGLE)
    
        # Oxygen temporarily has radical electron
        o_atom = emol.GetAtomWithIdx(o_idx)
        o_atom.SetNumRadicalElectrons(1)
    
        # Break C=O double bond → single
        bond = emol.GetBondBetweenAtoms(c_idx, o_idx)
        if bond and bond.GetBondType() == Chem.BondType.DOUBLE:
            bond.SetBondType(Chem.BondType.SINGLE)
    
        # Move radical from O → C
        o_atom.SetNumRadicalElectrons(0)   # oxygen satisfied
        c_atom = emol.GetAtomWithIdx(c_idx)
        c_atom.SetNumRadicalElectrons(1)   # carbonyl carbon gets radical

    # -------------------
    # Step 4: delocalize radicals via shortest paths
    # -------------------

    mol_with_radicals = emol.GetMol()
    radical_carbons = [a.GetIdx() for a in mol_with_radicals.GetAtoms()
                       if a.GetAtomicNum() == 6 and a.GetNumRadicalElectrons() > 0]

    conjugated_bonds = set()
    for bond in emol.GetBonds():
        begin = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()
        
        # Skip if not conjugated
        if not bond.GetIsConjugated():
            continue
    
        has_conjugated_carbon_neighbor = False
        for atom_idx in [begin, end]:
            atom = emol.GetAtomWithIdx(atom_idx)
            for nbr_bond in atom.GetBonds():
                if nbr_bond.GetIdx() == bond.GetIdx():
                    continue  # skip self
                
                if nbr_bond.GetIsConjugated():
                    nbr_atom = nbr_bond.GetOtherAtom(atom)
                    if nbr_atom.GetAtomicNum() == 6:  # must be carbon
                        has_conjugated_carbon_neighbor = True
                        break
            if has_conjugated_carbon_neighbor:
                break
    
        if has_conjugated_carbon_neighbor:
            conjugated_bonds.add((begin, end))
            conjugated_bonds.add((end, begin))

    def find_shortest_alternating_conjugated_path(mol, start_idx, end_idx):
        """
        Find the shortest path from start_idx to end_idx along carbons
        with alternating single/double bonds and optionally aromatic bonds.
        Avoid revisiting atoms.
        Returns a list of atom indices or None if no path exists.
        """
    
        all_paths = []
    
        def dfs(current_idx, expected_bond_types, path):
            if current_idx == end_idx:
                all_paths.append(path)
                return
    
            atom = mol.GetAtomWithIdx(current_idx)
            for nbr in atom.GetNeighbors():
                nbr_idx = nbr.GetIdx()
                bond = mol.GetBondBetweenAtoms(current_idx, nbr_idx)
                bond_type = bond.GetBondType()
    
                # Rule 1: only carbon
                if nbr.GetAtomicNum() != 6:
                    continue
    
                # Rule 2: bond type must match allowed types
                if expected_bond_types is not None and bond_type not in expected_bond_types:
                    continue
    
                # Rule 3: do not revisit atoms in current path
                if nbr_idx in path:
                    continue
    
                # Determine next expected bond types
                if bond_type == Chem.BondType.SINGLE:
                    next_expected = (Chem.BondType.DOUBLE, Chem.BondType.AROMATIC)
                elif bond_type == Chem.BondType.DOUBLE:
                    next_expected = (Chem.BondType.SINGLE, Chem.BondType.AROMATIC)
                elif bond_type == Chem.BondType.AROMATIC:
                    # after aromatic, allow SINGLE, DOUBLE, or stay aromatic
                    next_expected = (Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.AROMATIC)
    
                dfs(nbr_idx, next_expected, path + [nbr_idx])
    
        # Start DFS with expected bond type SINGLE or aromatic
        dfs(start_idx, (Chem.BondType.SINGLE, Chem.BondType.AROMATIC), [start_idx])
    
        if all_paths:
            # Return shortest path
            return min(all_paths, key=len)
        else:
            return None

    skip_molecule = False

    while len(radical_carbons) > 1:
        c1 = radical_carbons[0]
        c2 = radical_carbons[1]
    
        path = find_shortest_alternating_conjugated_path(emol, c1, c2)
        if not path:
            print(f"No conjugated path between radicals {c1} and {c2}, skipping molecule...")
            skip_molecule = True
            break  # stop processing this molecule entirely
    
        # Flip bonds along the path
        for i in range(len(path) - 1):
            b = emol.GetBondBetweenAtoms(path[i], path[i+1])
            if b.GetBondType() == Chem.BondType.SINGLE:
                b.SetBondType(Chem.BondType.DOUBLE)
            elif b.GetBondType() == Chem.BondType.DOUBLE:
                b.SetBondType(Chem.BondType.SINGLE)
    
        # Remove radicals and set final bond double
        emol.GetAtomWithIdx(c1).SetNumRadicalElectrons(0)
        emol.GetAtomWithIdx(c2).SetNumRadicalElectrons(0)
        b = emol.GetBondBetweenAtoms(c1, c2)
        if b:
            b.SetBondType(Chem.BondType.DOUBLE)
    
        radical_carbons = radical_carbons[2:]
    
    # -------------------
    # Finalize molecule
    # -------------------
    if skip_molecule:
        newmol = None
    else:
        newmol = emol.GetMol()
        if not len(get_valence_violations(mol)) == 0:
            print("⚠️ Warning: Some atoms have unusual valence after transformation")
    # -------------------
    # Draw before/after side by side
    # -------------------

    changed_carbon_indices = [c for c, o in carbonyls] if 'carbonyls' in locals() else []

    # Prepare highlight lists for the two molecules (same indices for both to show correspondences)
    highlight_atom_lists = [changed_carbon_indices, changed_carbon_indices]

    from rdkit.Chem.Draw import MolsToGridImage
    from IPython.display import display

    img = MolsToGridImage(
        [before, newmol],
        molsPerRow=2,
        subImgSize=(380, 380),
        legends=["Quinone (before)", "Hydroquinone (after)"],
        highlightAtomLists=highlight_atom_lists
    )

    # display(img)

    # Return the transformed molecule as before
    return newmol

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdmolops
from tqdm import tqdm

# Convert quinones to hydroquinones
# Only process rows in these groups
groups_to_process = ['Both', 'Carbonyl_in_ring']

# Initialize new column
Gao_Data_Raw['hydroquinone_smiles'] = None

for idx, row in tqdm(Gao_Data_Raw.iterrows(), total=len(Gao_Data_Raw)):
    if row['Group'] in groups_to_process:
        smi = row['canonical_SMILES']
        try:
            # Only take the molecule from the function, ignore SVG/drawing
            rwmol = reduce_quinone_stepwise_with_radicals(smi)

            # Convert RWMol to Mol before generating SMILES
            mol = Chem.Mol(rwmol)
            new_smi = Chem.MolToSmiles(mol, canonical=True)

            Gao_Data_Raw.at[idx, 'hydroquinone_smiles'] = new_smi

        except Exception as e:
            # Optionally, log failed SMILES
            print(f"Failed for index {idx}, SMILES={smi}: {e}")
            Gao_Data_Raw.at[idx, 'hydroquinone_smiles'] = None



In [ ]:
# Check on compounds that couldn't be processed
groups_to_process = ['Both', 'Carbonyl_in_ring']

# Filter rows where group is in groups_to_process and hydroquinone_smiles is None
failed_rows = Gao_Data_Raw[
    (Gao_Data_Raw['Group'].isin(groups_to_process)) &
    (Gao_Data_Raw['hydroquinone_smiles'].isna())
]

print(f"Number of molecules in {groups_to_process} with hydroquinone_smiles=None: {len(failed_rows)}")


from rdkit.Chem import Draw

# Take a small sample (say 6 molecules max)
sample_failed = failed_rows.sample(n=min(6, len(failed_rows)), random_state=42)

# Convert canonical_SMILES to Mol objects
mols = []
legends = []
for idx, row in sample_failed.iterrows():
    smi = row['canonical_SMILES']
    mol = Chem.MolFromSmiles(smi)
    if mol:  # only append valid mols
        mols.append(mol)
        legends.append(f"Idx {idx}")

# Visualize as grid
if mols:
    img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300,300), legends=legends)
    display(img)
else:
    print("No valid molecules to display.")

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from tqdm import tqdm
# Check for molecules with valence violations after transformation
# List to store indices of molecules that fail
failed_indices = []
failed_mols = []

for idx, smi in tqdm(enumerate(Gao_Data_Raw['hydroquinone_smiles']), total=len(Gao_Data_Raw)):
    if smi is None:
        continue
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            failed_indices.append(idx)
            failed_mols.append(mol)
    except Exception as e:
        failed_indices.append(idx)
        failed_mols.append(None)

print(f"Number of molecules with valence violations: {len(failed_indices)}")

failed_rw_mols = []

for idx in failed_indices[:13]:  # just take first 5 failures for visualization
    smi = Gao_Data_Raw.at[idx, 'canonical_SMILES']
    try:
        # regenerate the RWMol from your transformation function
        rw_mol = reduce_quinone_stepwise_with_radicals(smi)  # modify function to optionally suppress SVG
        failed_rw_mols.append(rw_mol)
    except Exception as e:
        print(f"Failed to regenerate RWMol for index {idx}, SMILES={smi}: {e}")
        failed_rw_mols.append(None)

from rdkit.Chem.Draw import MolsToGridImage

highlighted_mols = []
highlight_lists = []

for rw_mol in failed_rw_mols:
    if rw_mol:
        mol = Chem.Mol(rw_mol)  # convert RWMol back to Mol
        bad_atoms = get_valence_violations(mol)
        highlighted_mols.append(mol)
        highlight_lists.append(bad_atoms)

if highlighted_mols:
    img = MolsToGridImage(
        highlighted_mols,
        molsPerRow=3,
        subImgSize=(300, 300),
        highlightAtomLists=highlight_lists,
        legends=[f"Violations: {len(lst)}" for lst in highlight_lists]
    )
    display(img)
else:
    print("No molecules with valence violations to display.")


In [ ]:
import pandas as pd
from rdkit import Chem
# Keep only those molecules that could be successfully converted
# Make a copy first
Gao_Data_HQ = Gao_Data_Raw.copy()

# Step 1: keep only rows where hydroquinone_smiles is not None/NaN
Gao_Data_HQ = Gao_Data_HQ[Gao_Data_HQ['hydroquinone_smiles'].notna()].copy()

# Step 2: filter out molecules with valence issues
valid_indices = []
for idx, smi in Gao_Data_HQ['hydroquinone_smiles'].items():
    mol = Chem.MolFromSmiles(smi)
    if mol and len(get_valence_violations(mol)) == 0:
        valid_indices.append(idx)

# Keep only valid molecules
Gao_Data_HQ = Gao_Data_HQ.loc[valid_indices].reset_index(drop=True)

print(f"Number of valid hydroquinone molecules: {len(Gao_Data_HQ)}")

Gao_Data_HQ = Gao_Data_HQ.rename(columns={'canonical_SMILES': 'quinone_SMILES'})
Gao_Data_HQ = Gao_Data_HQ.rename(columns={'hydroquinone_smiles': 'canonical_SMILES'})


In [ ]:
# Rename Columns
Gao_Data_Raw['E1/2 (SHE) (V)'] = Gao_Data_Raw['Redox Potential(V)']
Gao_Data_HQ['E1/2 (SHE) (V)'] = Gao_Data_HQ['Redox Potential(V)']

Gao_Data_HQ.to_csv('../Datasets/Curated Dataset(s)/Gao_HQs_w_SMILES.csv')
Gao_Data_Raw.to_csv('../Datasets/Curated Dataset(s)/Gao_w_SMILES.csv')

# Combine Datasets (or Not)

In [ ]:
import pandas as pd
# Combine processed dataframes from multiple sources
df1 = pd.read_csv('../Datasets/Curated Dataset(s)/Weinberg_w_SMILES.csv')  # Weinberg
df2 = pd.read_csv('../Datasets/Curated Dataset(s)/Gao_HQs_w_SMILES.csv')       # Gao

# Add a source label column
df1['source'] = 'Weinberg'
df2['source'] = 'Gao'

# Concatenate
concatenated_df = pd.concat([df1[['canonical_SMILES', 'E1/2 (SHE) (V)', 'source']],
                             df2[['canonical_SMILES', 'E1/2 (SHE) (V)', 'source']]],
                            ignore_index=True)

print(concatenated_df.head())


In [ ]:
import pandas as pd
# From dataframes, keep the first entry of duplicates from a single source.
# If duplicates exist between sources, keep the Weinberg data
gao_unique = concatenated_df[concatenated_df['source'] == 'Gao'].drop_duplicates(
    subset='canonical_SMILES', keep='first'
)
weinberg_unique = concatenated_df[concatenated_df['source'] == 'Weinberg'].drop_duplicates(
    subset='canonical_SMILES', keep='first'
)

# Identify SMILES that appear in Weinberg
weinberg_smiles = set(weinberg_unique['canonical_SMILES'])

# Remove any Gao rows whose SMILES also appear in Weinberg
gao_unique_filtered = gao_unique[~gao_unique['canonical_SMILES'].isin(weinberg_smiles)]

# Combine
final_df = pd.concat([gao_unique_filtered, weinberg_unique], ignore_index=True)

print(f"✅ Final dataframe shape after deduplication: {final_df.shape}")
print(final_df.head())


In [ ]:
# Rename Columns and Save Combined Data to csv
final_df = final_df.rename(columns={'SMILES': 'SMILES_websearch'})
final_df = final_df.rename(columns={'canonical_SMILES': 'SMILES'})
final_df = final_df.rename(columns={'E1/2 (SHE) (V)': 'OP'})
df_subset = final_df[['SMILES', 'OP']]

df_subset.to_csv("../Datasets/Curated Dataset(s)/Small_Organic_SMILES_to_OP_no_splits.csv", index=False)